# Website Text Extraction — Colab demo

This demo installs the repository package and runs the same Selenium service as
`python run.py`. Chrome and optional NLP models start only when needed.

1. Add an `API_KEY` in **Colab Secrets** and grant the notebook access.
2. Run setup. Choose the desired repository branch with `REF`.
3. Use the local API, or run the optional tunnel cell to expose it with Bearer auth.
   Cloudflare answers tunnel requests that run longer than 125 seconds with HTTP 524,
   although the API accepts deadlines up to 600 seconds: keep `timeout_ms` below
   that limit through the tunnel and split large batches.

The notebook never prints the API key. Retrieve it from Colab Secrets for API
clients and send it in `Authorization: Bearer …`. Do not save it in notebook output.
Colab runs as root, so the demo explicitly opts out of Chrome sandboxing; use a
sandboxed non-root browser deployment for production. No performance claim is made
for Colab's changing resource allocations.

PDF/Office extras are installed. PII is optional: install `.[pii]` and the requested
spaCy model first. Missing models return an error with no unredacted page content.
See the repository README for options and the 0.3 response/migration contract.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import userdata

api_key = userdata.get('API_KEY')
if not isinstance(api_key, str) or not api_key.strip():
    raise RuntimeError('Set API_KEY in Colab Secrets before starting the service')
os.environ['API_KEY'] = api_key
os.environ.setdefault('HOST', '127.0.0.1')
os.environ.setdefault('PORT', '8000')
os.environ.setdefault('UVICORN_WORKERS', '1')
os.environ.setdefault('SELENIUM_MAX_POOL_SIZE', '1')
os.environ.setdefault('CONVERSION_WORKERS', '1')
os.environ.setdefault('MAX_CONCURRENT_REQUESTS', '4')
os.environ['SSRF_PROTECTION'] = 'true'
os.environ['ALLOW_INSECURE_SSL'] = 'false'
os.environ['SELENIUM_NO_SANDBOX'] = 'true'  # Colab's root-owned demo environment.

REF = 'main'  # Select the PR branch here when evaluating unmerged changes.
project = Path('/content/Website-Textextraction-Selenium')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch',
                    'https://github.com/janschachtschabel/Website-Textextraction-Selenium.git', str(project)], check=True)
else:
    print('Using the existing checkout; update it explicitly if needed.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(project) + '[documents]'], check=True)

chrome = shutil.which('google-chrome') or shutil.which('google-chrome-stable')
if not chrome:
    package = '/tmp/extraction-google-chrome.deb'
    urllib.request.urlretrieve('https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb', package)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', package], check=True)
    chrome = shutil.which('google-chrome') or shutil.which('google-chrome-stable')
os.environ['CHROME_BINARY'] = chrome

with open('/tmp/extraction-api.log', 'w') as log:
    api_process = subprocess.Popen([sys.executable, 'run.py'], cwd=project,
                                   stdout=log, stderr=subprocess.STDOUT)
import httpx
end = time.monotonic() + 60
while time.monotonic() < end:
    if api_process.poll() is not None:
        raise RuntimeError('API stopped; inspect /tmp/extraction-api.log')
    try:
        if httpx.get('http://127.0.0.1:8000/health', timeout=2).status_code == 200:
            print('API ready at http://127.0.0.1:8000 with Bearer authentication; /docs stays disabled while an API key is set - see the README for the request schema.')
            break
    except httpx.HTTPError:
        pass  # Startup may not have bound the socket yet.
    time.sleep(1)
else:
    raise RuntimeError('API startup deadline exceeded; inspect /tmp/extraction-api.log')


In [ ]:
# Optional public tunnel. Running this cell exposes the authenticated local API.
import re

if not os.environ.get('API_KEY'):
    raise RuntimeError('An API key is required for a public tunnel')
cloudflared = shutil.which('cloudflared')
if not cloudflared:
    cloudflared = '/tmp/extraction-cloudflared'
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    os.chmod(cloudflared, 0o700)
log_path = Path('/tmp/extraction-tunnel.log')
with log_path.open('w') as log:
    tunnel_process = subprocess.Popen([cloudflared, 'tunnel', '--url', 'http://127.0.0.1:8000'],
                                      stdout=log, stderr=subprocess.STDOUT)
end = time.monotonic() + 45
while time.monotonic() < end:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_path.read_text())
    if match:
        print('Public API:', match.group(0))
        print('Use Authorization: Bearer with your API_KEY from Colab Secrets.')
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError('Tunnel stopped; inspect /tmp/extraction-tunnel.log')
    time.sleep(1)
else:
    tunnel_process.terminate()
    raise RuntimeError('Tunnel startup deadline exceeded')
